In [1]:
import pandas as pd

df = pd.read_csv("../data/raw_pos_export.csv")

# same fix from Task 2 — parse each row's format individually instead of
# letting pandas infer one format for the whole column
df["ts_parsed"] = pd.to_datetime(df["transaction_ts"], format="mixed", errors="coerce")

# sort so rows that belong to the same real transaction end up adjacent —
# grouping logic below depends on this order
df = df.sort_values(["store_id", "register_id", "ts_parsed"]).reset_index(drop=True)

In [2]:
df.head

<bound method NDFrame.head of      row_id store_id register_id cashier_name       transaction_ts  \
0        24     ST01       REG-1  Alicia Wong           2025-03-03   
1        25     ST01       REG-1  Alicia Wong           2025-03-03   
2        26     ST01       REG-1  Alicia Wong           2025-03-03   
3        27     ST01       REG-1  Alicia Wong           2025-03-03   
4        30     ST01       REG-1    tom reyes    03-Mar-2025 12:15   
..      ...      ...         ...          ...                  ...   
126     104     ST99       REG-3  Maria Gomez    04-Mar-2025 00:33   
127     105     ST99       REG-3  Maria Gomez    04-Mar-2025 00:33   
128     106     ST99       REG-3  Maria Gomez    04-Mar-2025 00:33   
129     108     ST99       REG-3    Tom Reyes  03/04/2025 01:09 AM   
130     116     ST99       REG-3  Alicia Wong  03/04/2025 03:17 AM   

    invoice_number customer_ref customer_name_raw       customer_email_raw  \
0              NaN         C003        Priya Nair  

In [3]:
GAP_MINUTES = 2  # tune this if spot-checks below show over- or under-grouping

# time since the previous row within the same store + register + customer
df["prev_ts"] = df.groupby(["store_id", "register_id", "customer_ref"])["ts_parsed"].shift(1)
df["gap_minutes"] = (df["ts_parsed"] - df["prev_ts"]).dt.total_seconds() / 60

# a row starts a new transaction if there's no valid previous row (first row
# in its group), or too much time passed since the last one
df["new_transaction"] = df["prev_ts"].isna() | (df["gap_minutes"] > GAP_MINUTES)

# cumulative sum of the flag turns "is this a new transaction?" into a unique
# id per transaction — deliberately NOT using invoice_number as the key,
# since Task 2 showed it's missing most of the time and can even repeat
df["reconstructed_txn_id"] = df["new_transaction"].cumsum()

# how many transactions did the rule produce, as a first sanity check
df["reconstructed_txn_id"].nunique()

67

In [4]:
# isolate rows that DO have a real invoice_number — these are our ground truth
has_invoice = df[df["invoice_number"].notna() & (df["invoice_number"] != "")]

# for each real invoice number, count how many DISTINCT reconstructed
# transaction ids it got split across — should be 1 every time if the rule
# is working; anything higher means the rule wrongly split a real transaction
has_invoice.groupby("invoice_number")["reconstructed_txn_id"].nunique().sort_values(ascending=False).head(10)

invoice_number
INV-1000    2
INV-1004    2
INV-1009    2
INV-1002    1
INV-1001    1
INV-1003    1
INV-1005    1
INV-1007    1
INV-1006    1
INV-1008    1
Name: reconstructed_txn_id, dtype: int64

In [5]:
# pick 3 random reconstructed transactions to eyeball against the raw rows
sample_ids = df["reconstructed_txn_id"].drop_duplicates().sample(3, random_state=1)

for txn_id in sample_ids:
    # print each sampled group so you can read it like a human would —
    # does the customer, timestamp, and store/register make sense together?
    print(f"--- reconstructed_txn_id = {txn_id} ---")
    print(df[df["reconstructed_txn_id"] == txn_id][
        ["row_id", "store_id", "register_id", "customer_ref", "customer_name_raw",
         "ts_parsed", "product_name_raw", "qty", "invoice_number"]
    ])

--- reconstructed_txn_id = 63 ---
     row_id store_id register_id customer_ref customer_name_raw  \
124      88     ST99       REG-3         C006      Amir Hussain   

              ts_parsed product_name_raw  qty invoice_number  
124 2025-03-03 22:29:00      Wool Beanie    1            NaN  
--- reconstructed_txn_id = 28 ---
    row_id store_id register_id customer_ref customer_name_raw  \
58       7     ST03       REG-1         C006      Amir Hussain   
59       8     ST03       REG-1         C006      Amir Hussain   

             ts_parsed     product_name_raw  qty invoice_number  
58 2025-03-03 09:59:00  Notebook - Recycled    3            NaN  
59 2025-03-03 09:59:00     Blue T-Shirt (M)    1            NaN  
--- reconstructed_txn_id = 54 ---
     row_id store_id register_id customer_ref customer_name_raw  \
111      39     ST99       REG-2         C003        Priya Nair   
112      40     ST99       REG-2         C003        Priya Nair   

              ts_parsed product_name_r

In [6]:
transactions = df.groupby("reconstructed_txn_id").agg(
    store_id=("store_id", "first"),           # first is fine — same store for the whole group
    register_id=("register_id", "first"),
    customer_ref=("customer_ref", "first"),
    start_ts=("ts_parsed", "min"),             # earliest line in the transaction
    end_ts=("ts_parsed", "max"),               # latest line in the transaction
    n_lines=("row_id", "count"),               # how many line items got grouped together
    # keep any real invoice numbers seen, for traceability back to source —
    # even though we're not using invoice_number as the actual key
    invoice_numbers_seen=("invoice_number", lambda s: sorted(set(x for x in s if isinstance(x, str) and x != ""))),
).reset_index()

transactions.head(10)

,reconstructed_txn_id,store_id,register_id,customer_ref,start_ts,end_ts,n_lines,invoice_numbers_seen
0,1,ST01,REG-1,C003,2025-03-03 00:00:00,2025-03-03 00:00:00,4,[]
1,2,ST01,REG-1,C003,2025-03-03 12:15:00,2025-03-03 12:16:00,3,[INV-1001]
2,3,ST01,REG-1,NaN,2025-03-03 14:29:00,2025-03-03 14:29:00,1,[INV-1004]
3,4,ST01,REG-1,NaN,2025-03-03 14:29:00,2025-03-03 14:29:00,1,[INV-1004]
4,5,ST01,REG-1,C005,2025-03-03 17:08:38,2025-03-03 17:09:29,2,[]
5,6,ST01,REG-1,C001,2025-03-03 22:18:00,2025-03-03 22:18:00,2,[]
6,7,ST01,REG-2,NaN,2025-03-03 16:26:00,2025-03-03 16:26:00,1,[]
7,8,ST01,REG-2,NaN,2025-03-03 19:14:41,2025-03-03 19:14:41,1,[]
8,9,ST01,REG-3,C005,2025-03-03 00:00:00,2025-03-03 00:00:00,4,[]
9,10,ST01,REG-3,C005,2025-03-03 00:00:00,2025-03-03 00:00:00,7,"[INV-1007, INV-1013]"


In [7]:
# dropna=False keeps NaN customer_ref (walk-ins) as their own valid group,
# instead of silently excluding those rows from the shift() computation
df["prev_ts"] = df.groupby(["store_id", "register_id", "customer_ref"], dropna=False)["ts_parsed"].shift(1)
df["gap_minutes"] = (df["ts_parsed"] - df["prev_ts"]).dt.total_seconds() / 60
df["new_transaction"] = df["prev_ts"].isna() | (df["gap_minutes"] > GAP_MINUTES)
df["reconstructed_txn_id"] = df["new_transaction"].cumsum()

In [8]:
# after rebuilding transactions with the dropna=False fix, check for the
# OPPOSITE failure mode: a reconstructed transaction containing more than
# one distinct real invoice number — a sign of accidental over-merging,
# usually caused by coarse (date-only) timestamps collapsing to the same value
multi_invoice = transactions[transactions["invoice_numbers_seen"].apply(len) > 1]
multi_invoice

,reconstructed_txn_id,store_id,register_id,customer_ref,start_ts,end_ts,n_lines,invoice_numbers_seen
9,10,ST01,REG-3,C005,2025-03-03 00:00:00,2025-03-03 00:00:00,7,"[INV-1007, INV-1013]"
49,50,ST99,REG-1,C004,2025-03-04 01:58:51,2025-03-04 01:58:51,2,"[INV-1009, INV-1018]"


In [9]:
# confirm the walk-in / dropna=False fix resolved the earlier duplicate-row split —
# INV-1004 should now map to exactly 1 reconstructed_txn_id, not 2
has_invoice = df[df["invoice_number"].notna() & (df["invoice_number"] != "")]
has_invoice.groupby("invoice_number")["reconstructed_txn_id"].nunique().sort_values(ascending=False).head(5)

invoice_number
INV-1000    2
INV-1009    2
INV-1001    1
INV-1003    1
INV-1002    1
Name: reconstructed_txn_id, dtype: int64

In [10]:
# INV-1009 maps to 2 different reconstructed transactions — look at the raw rows
# to see exactly where the split/merge happened
df[df["invoice_number"] == "INV-1009"][
    ["row_id", "store_id", "register_id", "customer_ref", "ts_parsed",
     "reconstructed_txn_id", "product_name_raw", "qty"]
]

,row_id,store_id,register_id,customer_ref,ts_parsed,reconstructed_txn_id,product_name_raw,qty
99,62,ST99,REG-1,C003,2025-03-03 18:35:00,44,ceramic mug,2
100,64,ST99,REG-1,C003,2025-03-03 18:36:00,44,blue tshirt m,1
104,63,ST99,REG-1,C003,NaT,46,Canvas Tote Bag,2


In [11]:
# rows with no parseable timestamp can't be reliably time-ordered at all —
# their position after sort_values() is an artifact of how pandas handles
# NaT (sorted last), not their true original time, so whatever
# reconstructed_txn_id they land in is effectively a guess
df["needs_manual_review"] = df["ts_parsed"].isna()

df[df["needs_manual_review"]][
    ["row_id", "store_id", "register_id", "customer_ref", "invoice_number", "reconstructed_txn_id"]
]

,row_id,store_id,register_id,customer_ref,invoice_number,reconstructed_txn_id
104,63,ST99,REG-1,C003,INV-1009,46


In [12]:
# a transaction is "low confidence" if any of its lines had a missing timestamp
review_flag = df.groupby("reconstructed_txn_id")["needs_manual_review"].any()
transactions["needs_manual_review"] = transactions["reconstructed_txn_id"].map(review_flag)
transactions[transactions["needs_manual_review"]]

ValueError: Cannot mask with non-boolean array containing NA / NaN values

In [13]:
# find any reconstructed_txn_id values in transactions that don't exist in
# review_flag's index -- these are the ones causing the NaN
missing_ids = set(transactions["reconstructed_txn_id"]) - set(review_flag.index)
missing_ids

{64, 65, 66, 67}

In [14]:
# rebuild transactions fresh here so it's guaranteed to match the current df,
# no matter what any earlier cell in the notebook left behind
transactions = df.groupby("reconstructed_txn_id").agg(
    store_id=("store_id", "first"),
    register_id=("register_id", "first"),
    customer_ref=("customer_ref", "first"),
    start_ts=("ts_parsed", "min"),
    end_ts=("ts_parsed", "max"),
    n_lines=("row_id", "count"),
    invoice_numbers_seen=("invoice_number", lambda s: sorted(set(x for x in s if isinstance(x, str) and x != ""))),
).reset_index()

# now build the review flag from the same, current df
review_flag = df.groupby("reconstructed_txn_id")["needs_manual_review"].any()
transactions["needs_manual_review"] = transactions["reconstructed_txn_id"].map(review_flag)
transactions[transactions["needs_manual_review"]]

,reconstructed_txn_id,store_id,register_id,customer_ref,start_ts,end_ts,n_lines,invoice_numbers_seen,needs_manual_review
45,46,ST99,REG-1,C004,2025-03-04 01:58:51,2025-03-04 01:58:51,2,"[INV-1009, INV-1018]",True


In [15]:
# see the actual rows behind the one flagged transaction, rather than
# trusting the aggregated view — min()/max() silently skip NaT values,
# which can hide exactly the kind of problem we're flagging
df[df["reconstructed_txn_id"] == 46][
    ["row_id", "store_id", "register_id", "customer_ref", "customer_name_raw",
     "ts_parsed", "invoice_number", "product_name_raw", "qty"]
]

,row_id,store_id,register_id,customer_ref,customer_name_raw,ts_parsed,invoice_number,product_name_raw,qty
103,111,ST99,REG-1,C004,David Okafor,2025-03-04 01:58:51,INV-1018,Enamel Pin Set,2
104,63,ST99,REG-1,C003,Priya Nair,NaT,INV-1009,Canvas Tote Bag,2


In [16]:
# invoice_number confirms row_id 63 (blank timestamp) actually belongs with
# its two siblings under INV-1009 (rows 62, 64) — not with the unrelated
# David Okafor / INV-1018 transaction it landed in purely because NaT sorts
# to the end of the ST99/REG-1 partition, disconnected from its true neighbors
correct_id = df.loc[df["row_id"] == 62, "reconstructed_txn_id"].iloc[0]
df.loc[df["row_id"] == 63, "reconstructed_txn_id"] = correct_id

In [19]:

print(f"Sum of n_lines across transactions: {transactions['n_lines'].sum()}  (should equal total line items above)")

Sum of n_lines across transactions: 131  (should equal total line items above)
